# Pipeline smoke-test notebook

This notebook **exercises every function** in:

- `src/pipeline_functions.py`
- `src/pipeline.py` (executed as a script via `runpy` so `argparse` works)

It will:

1. Locate your repo root and add `src/` to `sys.path`
2. Load real data if present (`data/transcriptomics_filtered.joblib` and `data/proteomics_matched.joblib`), otherwise generate **synthetic** data and write those joblibs
3. Call:
   - `_select_features_percentile`
   - `_select_features_pvalue`
   - `cv_pipeline_fixed_percentile`
   - `cv_pipeline_fixed_p`
   - `cv_pipeline_fixed_features`
4. Run `src/pipeline.py` twice (once for each pipeline choice) and verify outputs were written.

> If your real dataset is very large, the notebook will **automatically sub-sample** for speed (configurable).


In [1]:
# --- Parameters (edit if you want) ---
SMOKE_TEST_MODE = True          # If True, sub-sample large real datasets for speed
SMOKE_N_SAMPLES = 369            # Used for sub-sampling or synthetic generation
SMOKE_N_FEATURES = 22614          # Used for sub-sampling or synthetic generation
SMOKE_N_PROTEINS = 1           # Synthetic proteins if generating Y

PROTEIN_TO_TEST = "A2M"          # e.g. "P0". If None, first protein column is used.
RANDOM_SEED = 42


In [9]:
import os, sys, runpy
from pathlib import Path
import numpy as np
import pandas as pd
from joblib import load, dump

np.random.seed(RANDOM_SEED)

def find_repo_root(start: Path) -> Path:
    """Walk upward until we find src/pipeline_functions.py."""
    cur = start.resolve()
    for _ in range(10):
        if (cur / "src" / "pipeline_functions.py").exists() and (cur / "src" / "pipeline.py").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError(
        "Could not find repo root containing src/pipeline_functions.py. " 
        "Run this notebook from within the repo."
    )

repo_root = find_repo_root(Path.cwd())
src_dir = repo_root / "src"
data_dir = repo_root / "data"
data_dir.mkdir(exist_ok=True, parents=True)

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"Repo root: {repo_root}")
print(f"src/:      {src_dir}")
print(f"data/:     {data_dir}")


Repo root: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\pipeline_proteins
src/:      C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\pipeline_proteins\src
data/:     C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\pipeline_proteins\data


,best_sample,best_abs_error,mse_all_others,params_dict,feat_coeffs_dict
0,MCF7_BREAST,0.001851,0.082078,"{'alpha': 0.1, 'l1_ratio': 0.5}","{'ABCC8': -0.0, 'ABHD14A-ACY1': -0.0, 'AC00207..."


In [3]:
# Import ALL functions from src/pipeline_functions.py
from pipeline_functions import (
    _select_features_percentile,
    _select_features_pvalue,
    cv_pipeline_fixed_percentile,
    cv_pipeline_fixed_p,
    cv_pipeline_fixed_features,
)

print("Imported functions from pipeline_functions.py:")
for fn in [
    _select_features_percentile,
    _select_features_pvalue,
    cv_pipeline_fixed_percentile,
    cv_pipeline_fixed_p,
    cv_pipeline_fixed_features,
]:
    print(" -", fn.__name__)


Imported functions from pipeline_functions.py:
 - _select_features_percentile
 - _select_features_pvalue
 - cv_pipeline_fixed_percentile
 - cv_pipeline_fixed_p
 - cv_pipeline_fixed_features


In [ ]:
# --- Load real data if available, otherwise generate synthetic data and save joblibs ---
transcript_path = data_dir / "transcriptomics_filtered.joblib"
protein_path = data_dir / "proteomics_matched.joblib"

def make_synthetic_XY(n_samples=SMOKE_N_SAMPLES, n_features=SMOKE_N_FEATURES, n_proteins=SMOKE_N_PROTEINS):
    samples = [f"S{i:03d}" for i in range(n_samples)]
    genes = [f"G{i:04d}" for i in range(n_features)]
    proteins = [f"P{i}" for i in range(n_proteins)]

    X = pd.DataFrame(
        np.random.normal(size=(n_samples, n_features)),
        index=samples,
        columns=genes,
    )

    # Create proteins as sparse linear combos of genes + noise (to make selection meaningful)
    Y = pd.DataFrame(index=samples, columns=proteins, dtype=float)
    for p in proteins:
        active = np.random.choice(genes, size=max(5, n_features // 40), replace=False)
        w = np.random.normal(scale=1.0, size=len(active))
        Y[p] = X[active].values @ w + np.random.normal(scale=0.5, size=n_samples)
    return X, Y

if transcript_path.exists() and protein_path.exists():
    print("Found joblib files. Loading...")
    X = load(transcript_path)
    Y = load(protein_path)

    # Basic sanity checks
    assert isinstance(X, pd.DataFrame) and isinstance(Y, pd.DataFrame)
    assert X.index.equals(Y.index), "X and Y must share identical sample index"

    print("Loaded shapes:")
    print("X:", X.shape, "(samples x genes)")
    print("Y:", Y.shape, "(samples x proteins)")

    if SMOKE_TEST_MODE:
        # Sub-sample to keep runtime under control on big datasets
        if X.shape[0] > SMOKE_N_SAMPLES:
            X = X.iloc[:SMOKE_N_SAMPLES].copy()
            Y = Y.iloc[:SMOKE_N_SAMPLES].copy()
        if X.shape[1] > SMOKE_N_FEATURES:
            X = X.iloc[:, :SMOKE_N_FEATURES].copy()
        print("After SMOKE_TEST_MODE sub-sampling:")
        print("X:", X.shape, "Y:", Y.shape)
else:
    print("Joblib files not found. Generating synthetic data and writing joblibs...")
    X, Y = make_synthetic_XY()
    dump(X, transcript_path)
    dump(Y, protein_path)
    print("Wrote:")
    print(" -", transcript_path)
    print(" -", protein_path)
    print("Synthetic shapes:")
    print("X:", X.shape, "Y:", Y.shape)

protein_name = PROTEIN_TO_TEST or Y.columns[0]
print("\nProtein chosen for tests:", protein_name)


Found joblib files. Loading...
Loaded shapes:
X: (369, 22614) (samples x genes)
Y: (369, 5113) (samples x proteins)
After SMOKE_TEST_MODE sub-sampling:
X: (369, 22614) Y: (369, 5113)

Protein chosen for tests: A2M


In [8]:
protein_list_path = os.path.join(data_dir, "protein_list.txt")
# Read protein list (strip whitespace + remove Windows CR)
with open(protein_list_path, "r", encoding="utf-8", errors="ignore") as f:
    proteins = [line.strip().replace("\r", "") for line in f if line.strip()]
y_cols = set(pd.Index(getattr(Y, "columns", Y.index)).astype(str))
proteins = pd.Index(proteins).astype(str)
missing = [p for p in proteins if p not in y_cols]

print(f"Proteins in list: {len(proteins)}")
print(f"Proteins found in Y: {len(proteins) - len(missing)}")
print(f"Missing: {len(missing)}")

if missing:
    print("\nFirst 50 missing protein names:")
    for p in missing[:50]:
        print(repr(p))
else:
    print("\n✅ All proteins in protein_list.txt are present in the proteomics dataframe.")

Proteins in list: 100
Proteins found in Y: 100
Missing: 0

✅ All proteins in protein_list.txt are present in the proteomics dataframe.


In [5]:
# --- 1) Exercise _select_features_percentile and _select_features_pvalue ---
y_series = Y[protein_name]

# These functions expect X_feat_by_sample shaped as: (features x samples)
X_feat_by_sample = X.T  # genes x samples

sel_pct = _select_features_percentile(
    X_feat_by_sample=X_feat_by_sample,
    y=y_series,
    feature_selector="anova",
    percentile=5,
)
sel_pv = _select_features_pvalue(
    X_feat_by_sample=X_feat_by_sample,
    y=y_series,
    feature_selector="anova",
    p_value=0.05,
)

print("_select_features_percentile picked", len(sel_pct), "features. Example:", sel_pct[:10])
print("_select_features_pvalue picked", len(sel_pv), "features. Example:", sel_pv[:10])


_select_features_percentile picked 1131 features. Example: ['A1BG', 'A2M', 'ABHD18', 'ABI2', 'ABO', 'AC000093.1', 'AC002074.1', 'AC003070.1', 'AC004687.1', 'AC005083.1']
_select_features_pvalue picked 8652 features. Example: ['A1BG', 'A1BG-AS1', 'A2M', 'AADAT', 'AAK1', 'AAMDC', 'AAMP', 'AAR2', 'AASS', 'AATBC']


In [6]:
# --- 2) Exercise cv_pipeline_fixed_percentile ---
param_grid = {'alpha': [0.1, 1.0, 10.0], 'l1_ratio': [0.1, 0.5, 0.9]}

raw_pct = cv_pipeline_fixed_percentile(
    model=None,
    param_grid=param_grid,
    X=X,
    y=Y,
    protein_name=protein_name,
    feature_selector="anova",
    n_splits=5,
    percentile=5,
    save_only_best=True
)

print("cv_pipeline_fixed_percentile results:", raw_pct.shape)
display(raw_pct.head())


cv_pipeline_fixed_percentile results: (1, 5)


,best_sample,best_abs_error,mse_all_others,params_dict,feat_coeffs_dict
0,SW620_LARGE_INTESTINE,0.002112,0.802894,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A2M': 0.04208602342632097, 'ABH..."


In [18]:
raw_pct["error"]=  abs(raw_pct["true"] - raw_pct["predicted"])
raw_pct.sort_values(by="error", ascending=True)


,LOO sample,true,predicted,params_dict,features_coef_dict,error
118,NCIH1793_LUNG,-0.214855,-0.213824,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.040138...",0.001031
80,HCC38_BREAST,0.028643,0.023254,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.040213...",0.005389
178,CORL23_LUNG,-0.839679,-0.831917,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.040268...",0.007761
4,CAL51_BREAST,0.274652,0.259378,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.040340...",0.015274
244,MSTO211H_PLEURA,0.127378,0.111512,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.040175...",0.015866
...,...,...,...,...,...,...
361,A172_CENTRAL_NERVOUS_SYSTEM,3.905544,0.812848,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.046044...",3.092696
128,NCIH1703_LUNG,3.077011,-0.252590,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.039877...",3.329602
321,NCIH441_LUNG,2.644637,-1.054394,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.042448...",3.699031
357,TUHR4TKB_KIDNEY,3.405674,-0.318283,"{'alpha': 0.1, 'l1_ratio': 0.1}","{'A1BG': 0.03289696150038594, 'A1BG-AS1': 0.0,...",3.723957


In [28]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

# 1) Find BEST row by absolute error (|true - predicted|)
tmp = raw_pct.copy()
tmp["abs_error"] = (tmp["true"].astype(float) - tmp["predicted"].astype(float)).abs()
best_idx = tmp["abs_error"].idxmin()
best_row = tmp.loc[best_idx]

print("Best row (lowest abs error):")
display(best_row.to_frame().T)

# 2) Extract coefficients from that single best-fold model
coef_dict = best_row["features_coef_dict"]
if not isinstance(coef_dict, dict) or len(coef_dict) == 0:
    raise ValueError("Best row has empty/invalid features_coef_dict")

genes = [g for g in coef_dict.keys() if g in X.columns]
w = pd.Series({g: coef_dict[g] for g in genes}, dtype=float)

print(f"Using {len(genes)} genes from best-fold model coefficients.")

# 3) Predict ALL samples using THIS single model's coefficients (no refit)
# prediction = X[genes] @ w   (intercept assumed 0)
pred = (X[genes].astype(float).values @ w.values)

# 4) Build results table + RMSE
y_true = Y[protein_name].loc[X.index].astype(float).values
res = pd.DataFrame({
    "sample": X.index,
    "true": y_true,
    "predicted": pred,
})
res["error"] = res["true"] - res["predicted"]
res["abs_error"] = res["error"].abs()

rmse = float(np.sqrt(mean_squared_error(res["true"].values, res["predicted"].values)))
print(f"Total RMSE using BEST-row coefficients on ALL samples: {rmse:.4f}")

display(res.sort_values("abs_error", ascending=False).head(25))


Best row (lowest abs error):


,LOO sample,true,predicted,params_dict,features_coef_dict,error,abs_error
118,NCIH1793_LUNG,-0.214855,-0.213824,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.040138...",0.001031,0.001031


Using 2262 genes from best-fold model coefficients.
Total RMSE using BEST-row coefficients on ALL samples: 1.0331


,sample,true,predicted,error,abs_error
321,NCIH441_LUNG,2.644637,-0.135273,2.779910,2.779910
318,NCIH1435_LUNG,-3.367386,-0.670518,-2.696868,2.696868
160,MDAMB231_BREAST,-2.466638,0.017225,-2.483863,2.483863
125,LCLC103H_LUNG,-2.119043,0.165423,-2.284466,2.284466
337,NCIH1155_LUNG,-1.803824,0.425424,-2.229248,2.229248
357,TUHR4TKB_KIDNEY,3.405674,1.223576,2.182098,2.182098
130,NCIH2009_LUNG,-2.446536,-0.290552,-2.155983,2.155983
12,WM2664_SKIN,-0.987087,1.155324,-2.142410,2.142410
126,LOXIMVI_SKIN,-1.438837,0.636432,-2.075269,2.075269
207,RH30_SOFT_TISSUE,5.051861,2.984699,2.067162,2.067162


In [19]:
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error

# ---- 1) Union of all features ever used in raw_pct ----
# (handles None / NaN safely)
def _keys_or_empty(d):
    return list(d.keys()) if isinstance(d, dict) else []

union_features = sorted(set().union(*raw_pct["features_coef_dict"].apply(_keys_or_empty)))
# keep only those that exist in X columns
union_features = [g for g in union_features if g in X.columns]

print(f"Union feature count (raw_pct): {len(union_features)}")

# ---- 2) Build fixed-features DataFrame (same union list for every sample) ----
features_union_df = pd.DataFrame(index=X.index)
features_union_df["features"] = [union_features] * len(features_union_df)

# ---- 3) Run LOO again using fixed features (no feature selection) ----
# Uses cv_pipeline_fixed_features defined/imported earlier
raw_union = cv_pipeline_fixed_features(
    model=None,
    param_grid=param_grid,
    X=X,
    y=Y,
    protein_name=protein_name,
    features=features_union_df,
    n_splits=5,      # change if you want (3 is faster)
)

# ---- 4) Add error + RMSE + show results ----
raw_union["error"] = raw_union["true"].astype(float) - raw_union["predicted"].astype(float)
rmse_union = float(np.sqrt(mean_squared_error(raw_union["true"].astype(float), raw_union["predicted"].astype(float))))

print(f"raw_union shape: {raw_union.shape}")
print(f"LOO RMSE (fixed union features): {rmse_union:.4f}")

display(raw_union.head())  # table like raw_pct (includes params_dict + features_coef_dict)

# Optional: show worst absolute errors
display(raw_union.assign(abs_error=raw_union["error"].abs()).sort_values("abs_error", ascending=False).head(10))

# Optional: coefficient table (wide) for inspection (can be big!)
# coef_wide = pd.DataFrame(raw_union["features_coef_dict"].tolist()).fillna(0.0)
# display(coef_wide.head())


Union feature count (raw_pct): 2873
raw_union shape: (369, 7)
LOO RMSE (fixed union features): 1.0273


,protein,LOO sample,true,predicted,params_dict,features_coef_dict,error
0,A2M,MDAMB468_BREAST,-0.345562,-0.516870,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.041094...",0.171308
1,A2M,SH4_SKIN,1.642186,0.521712,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.037098...",1.120474
2,A2M,AU565_BREAST,-1.129114,-0.844703,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.041440...",-0.284411
3,A2M,KMRC1_KIDNEY,1.493333,0.030410,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.042004...",1.462923
4,A2M,CAL51_BREAST,0.274652,0.246714,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.041451...",0.027938


,protein,LOO sample,true,predicted,params_dict,features_coef_dict,error,abs_error
207,A2M,RH30_SOFT_TISSUE,5.051861,1.013118,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.043234...",4.038743,4.038743
321,A2M,NCIH441_LUNG,2.644637,-1.036547,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.043170...",3.681184,3.681184
357,A2M,TUHR4TKB_KIDNEY,3.405674,-0.148937,"{'alpha': 0.1, 'l1_ratio': 0.1}","{'A1BG': 0.02245939559776068, 'A1BG-AS1': 0.0,...",3.554611,3.554611
128,A2M,NCIH1703_LUNG,3.077011,-0.266862,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.040314...",3.343874,3.343874
361,A2M,A172_CENTRAL_NERVOUS_SYSTEM,3.905544,0.832357,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.046695...",3.073187,3.073187
343,A2M,C32_SKIN,3.455395,0.477794,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.034591...",2.977602,2.977602
214,A2M,KU1919_URINARY_TRACT,2.748382,-0.205463,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.041680...",2.953845,2.953845
240,A2M,KNS42_CENTRAL_NERVOUS_SYSTEM,3.654857,0.720480,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.023108...",2.934378,2.934378
334,A2M,G402_SOFT_TISSUE,3.040368,0.450146,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.042765...",2.590222,2.590222
164,A2M,LN229_CENTRAL_NERVOUS_SYSTEM,3.285918,0.711196,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.038315...",2.574722,2.574722


In [20]:
display(raw_union.assign(abs_error=raw_union["error"].abs()).sort_values("abs_error", ascending=True).head(10))


,protein,LOO sample,true,predicted,params_dict,features_coef_dict,error,abs_error
178,A2M,CORL23_LUNG,-0.839679,-0.835283,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.041341...",-0.004396,0.004396
362,A2M,BT20_BREAST,-0.649348,-0.644146,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.041227...",-0.005202,0.005202
70,A2M,LS513_LARGE_INTESTINE,-0.903152,-0.909529,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.041379...",0.006377,0.006377
80,A2M,HCC38_BREAST,0.028643,0.019187,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.041313...",0.009456,0.009456
118,A2M,NCIH1793_LUNG,-0.214855,-0.225191,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.041248...",0.010336,0.010336
193,A2M,TE4_OESOPHAGUS,-0.630793,-0.619228,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.041231...",-0.011565,0.011565
223,A2M,HEC265_ENDOMETRIUM,-0.638222,-0.625757,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.041342...",-0.012465,0.012465
266,A2M,BXPC3_PANCREAS,-0.784391,-0.768331,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.041413...",-0.016060,0.016060
244,A2M,MSTO211H_PLEURA,0.127378,0.110685,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.041293...",0.016693,0.016693
140,A2M,SW948_LARGE_INTESTINE,-0.873798,-0.892553,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.041427...",0.018755,0.018755


In [27]:
import numpy as np
import pandas as pd

# ✅ Set this to the dataframe you displayed in the screenshot
df = raw_union.assign(abs_error=raw_union["error"].abs()).sort_values("abs_error", ascending=True)

# Tissue is everything AFTER the first underscore in "LOO sample"
# e.g. "A172_CENTRAL_NERVOUS_SYSTEM" -> "CENTRAL_NERVOUS_SYSTEM"
df2 = df.copy()
df2["tissue"] = df2["LOO sample"].astype(str).str.split("_", n=1).str[1]

# MSE per tissue = mean((true - predicted)^2)
df2["sq_error"] = (df2["true"].astype(float) - df2["predicted"].astype(float)) ** 2

mse_by_tissue = (
    df2.groupby("tissue", as_index=False)
       .agg(
           n=("sq_error", "size"),
           mse=("sq_error", "mean"),
           rmse=("sq_error", lambda x: float(np.sqrt(np.mean(x))))  # optional but useful
       )
       .sort_values("mse", ascending=False)
)

mse_by_tissue


,tissue,n,mse,rmse
17,SOFT_TISSUE,8,4.251995,2.062037
4,CENTRAL_NERVOUS_SYSTEM,13,2.699239,1.642936
7,KIDNEY,11,2.454248,1.566604
19,THYROID,2,2.016634,1.420082
0,AUTONOMIC_GANGLIA,1,1.507957,1.227989
21,URINARY_TRACT,11,1.426384,1.194313
10,LUNG,75,1.332912,1.154518
2,BONE,7,1.305546,1.142605
9,LIVER,14,1.139713,1.067574
12,OVARY,17,1.055691,1.027468


In [26]:
len(df2)

10

In [7]:
import numpy as np, pandas as pd

df = raw_pct.copy()

# ----- FEATURES: stability + effect sizes -----
coefs = pd.DataFrame(df["features_coef_dict"].tolist()).fillna(0.0)
EPS = 1e-10
sel = coefs.abs() > EPS

feat_summary = pd.DataFrame({
    "freq_selected": sel.mean(0),
    "n_selected": sel.sum(0).astype(int),
    "mean_coef": coefs.mean(0),
    "mean_coef_selected": coefs.where(sel).mean(0),
    "mean_abs_coef_selected": coefs.abs().where(sel).mean(0),
}).sort_values(["freq_selected","mean_abs_coef_selected"], ascending=[False, False])

print("Always chosen:", int((feat_summary["freq_selected"] == 1.0).sum()))
print(">=80% chosen:", int((feat_summary["freq_selected"] >= 0.8).sum()))
display(feat_summary.head(30))  # most stable genes

# Optional: lists
always_genes = feat_summary.index[feat_summary["freq_selected"] == 1.0].tolist()
top80_genes  = feat_summary.index[feat_summary["freq_selected"] >= 0.8].tolist()

# ----- HYPERPARAMS: frequency + performance -----
params = df["params_dict"].apply(pd.Series)
combo_counts = params.value_counts().sort_values(ascending=False)
print("\nTop hyperparam combos (count):")
display(combo_counts)

err = (df["true"].astype(float) - df["predicted"].astype(float))
tmp = df.join(params)
tmp["mse"] = err**2
tmp["abs_err"] = err.abs()

perf_by_combo = (
    tmp.groupby(["alpha","l1_ratio"])
       .agg(n=("mse","size"),
            rmse=("mse", lambda x: float(np.sqrt(np.mean(x)))),
            mae=("abs_err","mean"))
       .sort_values("rmse")
)
print("\nHyperparam combos ranked by RMSE:")
display(perf_by_combo)


Always chosen: 56
>=80% chosen: 120


,freq_selected,n_selected,mean_coef,mean_coef_selected,mean_abs_coef_selected
IFIH1,1.0,369,-0.041541,-0.041541,0.041541
A2M,1.0,369,0.041386,0.041386,0.041386
LRP1,1.0,369,0.041384,0.041384,0.041384
PCDHGB7,1.0,369,0.036917,0.036917,0.036917
DIRC3,1.0,369,0.035764,0.035764,0.035764
CPA2,1.0,369,0.034016,0.034016,0.034016
AC244517.6,1.0,369,0.029986,0.029986,0.029986
SCARF2,1.0,369,0.029984,0.029984,0.029984
SAMD10,1.0,369,-0.028869,-0.028869,0.028869
PHPT1,1.0,369,0.028082,0.028082,0.028082



Top hyperparam combos (count):


alpha  l1_ratio
1.0    0.1         369
Name: count, dtype: int64


Hyperparam combos ranked by RMSE:


,,n,rmse,mae
alpha,l1_ratio,,,
1.0,0.1,369,1.043794,0.768816


In [9]:
# --- 3) Exercise cv_pipeline_fixed_p ---
raw_p = cv_pipeline_fixed_p(
    model=None,
    param_grid=param_grid,
    X=X,
    y=Y,
    protein_name=protein_name,
    feature_selector="anova",
    n_splits=5,
    p_value=0.05,
)

print("cv_pipeline_fixed_p results:", raw_p.shape)
display(raw_p.head())

mse_p = float(np.mean((raw_p['true'].values - raw_p['predicted'].values) ** 2))
rmse_p = float(np.sqrt(mse_p))
print(f"LOO RMSE (fixed_p): {rmse_p:.4f}")


cv_pipeline_fixed_p results: (369, 5)


,LOO sample,true,predicted,params_dict,features_coef_dict
0,MDAMB468_BREAST,-0.345562,-0.453095,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.038680..."
1,SH4_SKIN,1.642186,0.503482,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.034480..."
2,AU565_BREAST,-1.129114,-0.833973,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.038962..."
3,KMRC1_KIDNEY,1.493333,0.095808,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.039127..."
4,CAL51_BREAST,0.274652,0.245218,"{'alpha': 1.0, 'l1_ratio': 0.1}","{'A1BG': 0.0, 'A1BG-AS1': 0.0, 'A2M': 0.038855..."


LOO RMSE (fixed_p): 1.0354


In [11]:
# --- 4) Exercise cv_pipeline_fixed_features ---
# Build a per-sample fixed feature list (here: top correlated genes with the target, computed globally for simplicity)
corr = X.apply(lambda col: np.corrcoef(col.values, y_series.values)[0, 1])
corr = corr.fillna(0.0).abs().sort_values(ascending=False)

top_k = max(10, min(30, len(corr)//5))
top_genes = corr.index[:top_k].tolist()

features_df = pd.DataFrame(index=X.index)
features_df['features'] = [top_genes[:] for _ in range(len(features_df))]  # same list for each sample (smoke test)

raw_fx = cv_pipeline_fixed_features(
    model=None,
    param_grid=param_grid,
    X=X,
    y=Y,
    protein_name=protein_name,
    features=features_df,
    n_splits=5,
)

print("cv_pipeline_fixed_features results:", raw_fx.shape)
display(raw_fx.head())

mse_fx = float(np.nanmean((raw_fx['true'].values - raw_fx['predicted'].values) ** 2))
rmse_fx = float(np.sqrt(mse_fx))
print(f"LOO RMSE (fixed_features): {rmse_fx:.4f}")


cv_pipeline_fixed_features results: (369, 6)


,protein,LOO sample,true,predicted,params_dict,features_coef_dict
0,A2M,MDAMB468_BREAST,-0.345562,-0.435594,"{'alpha': 0.1, 'l1_ratio': 0.1}","{'SCARF2': 0.10382945763656624, 'LRP1': 0.1383..."
1,A2M,SH4_SKIN,1.642186,0.622572,"{'alpha': 0.1, 'l1_ratio': 0.1}","{'SCARF2': 0.1016648775422842, 'LRP1': 0.13593..."
2,A2M,AU565_BREAST,-1.129114,-1.115096,"{'alpha': 0.1, 'l1_ratio': 0.1}","{'SCARF2': 0.10355978560996888, 'LRP1': 0.1377..."
3,A2M,KMRC1_KIDNEY,1.493333,0.134858,"{'alpha': 0.1, 'l1_ratio': 0.1}","{'SCARF2': 0.10531142321353991, 'LRP1': 0.1363..."
4,A2M,CAL51_BREAST,0.274652,0.176916,"{'alpha': 0.1, 'l1_ratio': 0.1}","{'SCARF2': 0.10377924861580665, 'LRP1': 0.1381..."


LOO RMSE (fixed_features): 0.9876


In [7]:
# --- 5) Execute src/pipeline.py end-to-end (this exercises select_pipeline + script flow) ---
# We'll run it twice: fixed_percentile and fixed_p.
# Note: pipeline.py will read the joblibs from data/ we ensured above.

pipeline_py = src_dir / "pipeline.py"
assert pipeline_py.exists(), f"Missing {pipeline_py}"

def run_pipeline_script(protein: str, pipeline: str):
    old_argv = sys.argv[:]
    try:
        sys.argv = [str(pipeline_py), '--protein', protein, '--pipeline', pipeline, '--model', 'elasticnet']
        print("\nRunning:", " ".join(sys.argv))
        g = runpy.run_path(str(pipeline_py), run_name='__main__')
        return g
    finally:
        sys.argv = old_argv

globals_fixed_pct = run_pipeline_script(protein_name, 'fixed_percentile')
#globals_fixed_p = run_pipeline_script(protein_name, 'fixed_p')

print("\nScript runs completed. Checking for outputs in data/ ...")
for p in sorted(data_dir.glob(f"{protein_name}_*")):
    print(" -", p.name)



Running: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\pipeline_proteins\src\pipeline.py --protein A2M --pipeline fixed_percentile --model elasticnet
Loading Data...
Starting LOO for A2M using pipeline=fixed_percentile ...
Done. Saved: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\pipeline_proteins\data\A2M_best.parquet

Script runs completed. Checking for outputs in data/ ...
 - A2M_best.parquet


In [ ]:
# --- 6) Explicitly call select_pipeline from pipeline.py globals to 'try' it directly ---
# select_pipeline is defined in pipeline.py; runpy returned it in the globals dict.
select_pipeline = globals_fixed_pct.get('select_pipeline')
assert callable(select_pipeline), "select_pipeline not found in pipeline.py globals"

print("select_pipeline('fixed_percentile') ->", select_pipeline('fixed_percentile').__name__)
print("select_pipeline('fixed_p') ->", select_pipeline('fixed_p').__name__)

try:
    select_pipeline('not_a_real_pipeline')
except Exception as e:
    print("select_pipeline invalid input raised (expected):", repr(e))


---

## Done

If everything ran without exceptions, you've successfully **smoke-tested every function** in:

- `src/pipeline_functions.py`
- `src/pipeline.py` (including its CLI-style execution)

You can now swap `SMOKE_TEST_MODE=False` to run on the full dataset (warning: it may take a while depending on size).
